:::{danger} Not published: the headline is not identified by this design

This page is withheld from the book. Its title says the break *raised*
returns, and its prose calls baseline variables confounders, post-break
variables mediators, and fitted contrasts effects. None of that is
supported by what it does:

- The break date is chosen after seeing the data, so the comparison is
  between periods selected partly for differing.
- The panel holds stocks in the index *today*, so firms that left it are
  absent from the pre-break window they belonged to.
- There is no parallel-trends argument, no comparison group that was not
  exposed, and no sensitivity analysis.
- Half-samples are drawn over rows, ignoring that returns are clustered
  within a stock and correlated across stocks on the same day, so the
  resampling understates uncertainty.
- The claim that thinning to every third day costs only power, and that a
  term clearing there would clear on the full panel, is asserted rather
  than shown.

Two revisions would make it publishable. The **descriptive** one keeps the
date and the tiers as descriptions, uses blocked or clustered validation and
stability resampling, drops effect/mediator/confounder/intent-to-treat
language, and makes no claim from the thinned sample to the full panel. The
**causal** one needs a pre-specified design, stated identification
assumptions, a defensible comparison group, cluster-aware uncertainty, and
sensitivity analysis. Until one is done, the analysis below is kept for its
method rather than its conclusions.
:::

# Returns and dispersion across the AI adoption break

On any recent trading day, an artificial-intelligence leader and a typical
S&P 500 stock lived in different markets. One hypothesis for what has changed
since late 2022 holds that the index now runs in two lanes. In the first,
stocks tied to AI — the chipmakers and software firms building it, the power
producers and data-centre landlords housing it — moved to a higher level of
expected returns while the rest of the index lagged behind. In the second,
the dispersion of individual outcomes widened: day-to-day results spread out
more across stocks than they used to, and the widening may be a step rather
than a drift.

Both claims are statements about parameters of a return distribution other
than its centre. Comparing mean returns cannot see the dispersion claim at
all, and a volatility computed on the whole index cannot tell an AI-specific
widening from a market-wide turbulent stretch. This case study fits one
distributional regression that estimates both lanes simultaneously: a
location model for expected abnormal returns and a scale model for their
dispersion, each with its own variable competition, using `gamboostLSS`
{cite:t}`hofner2014gamboostlss` on roughly nine hundred thousand stock-day
observations of S&P 500 constituents from January 2019 to August 2026.

In [ ]:
from collections import Counter
from datetime import date
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
import polars as pl

import mboostrs as mb

# The cumulative-return chart has one point per trading day and tier, about
# 5,800 rows, above the 5,000 that Altair embeds by default.
alt.data_transformers.disable_max_rows()


def data_path(name):
    """Resolve book data under MyST and repository-root execution."""
    for directory in (Path("../data"), Path("data"), Path("docs/book/data")):
        candidate = directory / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(name)


BREAK = "2022-11-30"  # ChatGPT release date, the hypothesised regime break

TIER_LABEL = {
    0.0: "rest of index",
    0.5: "buildout adjacent",
    1.0: "AI core",
}

stored = pl.read_parquet(data_path("sp500_daily.parquet")).sort("ticker", "date")

# `vol60` and `beta252` are stored on rolling windows that end at day t and
# therefore contain day t's own return. Each stock's previous trading day's
# value is the backward-looking one, and is what the model may condition on.
lagged = stored.with_columns(
    pl.col("vol60").shift(1).over("ticker").alias("vol60_lag"),
    pl.col("beta252").shift(1).over("ticker").alias("beta252_lag"),
).drop_nulls(["vol60_lag", "beta252_lag"])

# Keep firms quoted on both sides of the break, so the before/after comparison
# is within-firm rather than partly a change in which firms are present.
spanning = (
    lagged.group_by("ticker")
    .agg((pl.col("post_chatgpt") == 0).any().alias("pre"),
         (pl.col("post_chatgpt") == 1).any().alias("post"))
    .filter(pl.col("pre") & pl.col("post"))
    .select("ticker")
)
late_entrants = (
    lagged.join(spanning, on="ticker", how="anti")
    .group_by("ticker").agg(pl.col("ai_score").first())
    .sort("ticker")
)

panel = (
    lagged.join(spanning, on="ticker", how="semi")
    .with_columns(
        ((pl.col("ret") - pl.col("beta252_lag") * pl.col("mkt")) * 100)
        .alias("xret_pct"),
        ((pl.col("date").cast(pl.Date) - pl.date(2019, 1, 1)).dt.total_days() // 28)
        .alias("block_idx"),
        (pl.col("ai_score") == 1.0).cast(pl.Float64).alias("core"),
        (pl.col("ai_score") == 0.5).cast(pl.Float64).alias("adj"),
    )
    .with_columns(
        (pl.col("core") * pl.col("post_chatgpt")).alias("core_post"),
        (pl.col("adj") * pl.col("post_chatgpt")).alias("adj_post"),
    )
)

# Each smooth candidate is split into a linear part and a centred non-linear
# part so that every base learner in the competition carries the same two
# degrees of freedom. The linear halves read duplicate columns because a
# learner is named for the column it reads, and two learners on one column
# would report under one name.
panel = panel.with_columns(
    pl.col("mom121").alias("mom_lin"),
    pl.col("vol60_lag").alias("vol_lin"),
    pl.col("block_idx").cast(pl.Float64).alias("block_lin"),
)

# Baseline versions of the two firm-level controls: each stock's own value on
# the last trading day before the break, held constant over its whole history.
# Fixed before the break, so nothing the break did can move them.
baseline = (
    panel.filter(pl.col("post_chatgpt") == 0)
    .sort("date")
    .group_by("ticker")
    .agg(pl.col("vol60_lag").last().alias("vol_base"),
         pl.col("mom121").last().alias("mom_base"))
)
panel = panel.join(baseline, on="ticker", how="left").with_columns(
    pl.col("vol_base").alias("vol_base_lin"),
    pl.col("mom_base").alias("mom_base_lin"),
)

# How much of the stored covariate was the response itself: the correlation
# between trailing volatility and the size of the return it would explain.
leak_corr = float(np.corrcoef(np.abs(panel["xret_pct"]), panel["vol60"])[0, 1])
lag_corr = float(np.corrcoef(np.abs(panel["xret_pct"]), panel["vol60_lag"])[0, 1])

pdf = panel.to_pandas()
n_dropped_late = int(len(late_entrants))

## The data

The extract is built by `tools/curate_sp500_daily.py`: current S&P 500
constituents and GICS classifications from Wikipedia, adjusted daily closes
from Yahoo Finance (so the log difference is the total return including
dividends), and four covariates:

* `mom121` — momentum, the log price change from 252 to 21 trading days ago;
* `vol60` — annualised standard deviation of sixty daily returns;
* `beta252` — one-year market-model beta against the equal-weighted
  constituent index;
* `post_chatgpt` — zero before 2022-11-30, one afterwards.

`vol60` and `beta252` are stored on windows that **end on day $t$ inclusive**,
so each contains the return it would be used to explain. Both are replaced
with the same stock's previous trading day's value. The correction is not
cosmetic. The correlation between `vol60` and the absolute response is
{eval}`leak_corr:.3f` as stored and {eval}`lag_corr:.3f` once lagged, and
the difference is the covariate reading part of its own answer. `beta252`
enters the response itself, so without the lag the hedge would be fitted on a
window containing the return it removes. `mom121` needs no lag; its window already
closes twenty-one trading days before $t$.

The AI-exposure score assigns each stock a tier from its GICS sub-industry:

* **AI core** (`ai_score = 1`) — semiconductors, software, IT services, and
  technology hardware: firms whose revenue tracks spending on AI compute
  directly.
* **Buildout adjacent** (`ai_score = 0.5`) — interactive media platforms,
  networking and electronic equipment, electrical equipment, independent
  power producers, and data-centre REITs: firms selling power, connectivity,
  or physical capacity into the same buildout.
* **Rest of index** (`ai_score = 0`) — everything else.

The assignment is a judgment call applied uniformly to sub-industry labels,
and the full mapping stays auditable in the file's `sub_industry` column. The
score is used only to sort stocks into three groups; nothing below assumes
that the middle tier sits numerically half way between the other two.

In [ ]:
tiers = (
    panel.group_by("ai_score", "sub_industry")
    .agg(pl.col("ticker").n_unique().alias("n"))
    .sort("ai_score", "n", descending=[False, True])
)
for score in (1.0, 0.5):
    sub = tiers.filter(pl.col("ai_score") == score)
    top_rows = list(sub.sort("n", descending=True).iter_rows(named=True))[:4]
    names = ", ".join(f"{r['sub_industry']} ({r['n']})" for r in top_rows)
    print(f"{TIER_LABEL[score]:18s} {int(sub['n'].sum()):3d} tickers"
          f" — largest sub-industries: {names}")

n_tickers = panel["ticker"].n_unique()
n_days = panel["date"].n_unique()
density = panel.height / (n_tickers * n_days)
print(f"\n{n_tickers} tickers over {n_days} trading days: {panel.height:,} "
      f"stock-day rows, {density:.1%} of a balanced panel")
print(f"dropped for having no pre-break quotes: {n_dropped_late} tickers "
      f"({', '.join(late_entrants['ticker'].to_list())})")

ai_share = float(
    panel.filter(pl.col("ai_score") > 0)["ticker"].n_unique() / n_tickers)
print(f"the two AI tiers are {ai_share:.1%} of the constituent list")

Three limitations follow from how the panel is built.

**Survivorship.** The constituent list is today's index projected backwards.
Companies that dropped out of the S&P 500 since 2019 are absent entirely,
which removes the worst long-run losers from the rest-of-index lane and
biases its measured performance **upward** — against the two-lane hypothesis,
so the raw comparison understates rather than inflates the contrast.

**Composition.** The dropped late entrants are not a random sample: they
include GE Vernova, Constellation Energy and SanDisk, exactly the
power-for-data-centres and memory names the AI tiers would otherwise gain
only after the break. Restricting to firms quoted on both sides keeps the
comparison within-firm, at the cost of excluding the newest members of the
lanes under study.

**Exposure is industry-level**, not firm-level: Nvidia and a hypothetical
semiconductor firm with no AI business receive the same score, which works
against sharp lane separation.

## The raw returns

Before any modelling, the two claims in their unadjusted form: average
abnormal returns and cross-sectional dispersion, by era and tier. Dispersion
here is the average, over trading days in the era, of the cross-sectional
standard deviation of `xret_pct` across stocks, annualised.

One property of the response governs how the table reads. `xret_pct` is
hedged against the equal-weighted mean of the same constituents, so on every
day the cross-sectional average of the response is pinned near zero. The
lanes therefore sum to zero by construction: any tier that reads positive
forces the complementary tier negative. A negative number in the
rest-of-index row is not evidence that those stocks lost money; it says they
trailed the equal-weighted index, which is the comparison being asked for.

In [ ]:
era_label = (
    pl.when(pl.col("post_chatgpt") > 0)
    .then(pl.lit("Dec 2022 - Aug 2026"))
    .otherwise(pl.lit("Jan 2019 - Nov 2022"))
    .alias("era")
)

raw_means = (
    panel.with_columns(era_label)
    .group_by("era", "ai_score")
    .agg((pl.col("xret_pct").mean() * 252).alias("ann_abnormal_pp"))
)

raw_disp = (
    panel.with_columns(era_label)
    .group_by("era", "date", "ai_score")
    .agg(pl.col("xret_pct").std().alias("day_sd"))
    .group_by("era", "ai_score")
    .agg((pl.col("day_sd").mean() * np.sqrt(252)).alias("ann_dispersion_pp"))
)

raw_table = (
    raw_means.join(raw_disp, on=["era", "ai_score"])
    .with_columns(
        pl.col("ai_score").replace_strict(TIER_LABEL).alias("tier"),
        pl.col("ann_abnormal_pp").round(1),
        pl.col("ann_dispersion_pp").round(1),
    )
    .sort("era", "ai_score")
    .select("era", "tier", "ann_abnormal_pp", "ann_dispersion_pp")
)
display(raw_table)


def raw_cell(era_prefix, score, column):
    row = (
        raw_means.join(raw_disp, on=["era", "ai_score"])
        .filter(pl.col("era").str.starts_with(era_prefix),
                pl.col("ai_score") == score)
    )
    return float(row[column][0])


raw = {(era, score): {c: raw_cell(era, score, c)
                      for c in ("ann_abnormal_pp", "ann_dispersion_pp")}
       for era in ("Jan", "Dec") for score in (0.0, 0.5, 1.0)}

core_gap_raw_pre = raw[("Jan", 1.0)]["ann_abnormal_pp"] - raw[("Jan", 0.0)]["ann_abnormal_pp"]
core_gap_raw_post = raw[("Dec", 1.0)]["ann_abnormal_pp"] - raw[("Dec", 0.0)]["ann_abnormal_pp"]
adj_gap_raw_pre = raw[("Jan", 0.5)]["ann_abnormal_pp"] - raw[("Jan", 0.0)]["ann_abnormal_pp"]
adj_gap_raw_post = raw[("Dec", 0.5)]["ann_abnormal_pp"] - raw[("Dec", 0.0)]["ann_abnormal_pp"]
core_disp_raw = raw[("Dec", 1.0)]["ann_dispersion_pp"] / raw[("Jan", 1.0)]["ann_dispersion_pp"]
print(f"AI core minus rest:           {core_gap_raw_pre:+.1f} pp/yr before,"
      f" {core_gap_raw_post:+.1f} after")
print(f"buildout adjacent minus rest: {adj_gap_raw_pre:+.1f} pp/yr before,"
      f" {adj_gap_raw_post:+.1f} after")
print(f"AI-core dispersion after/before: {core_disp_raw:.2f}")

Two features of the raw table shape everything that follows. The AI-core
lane earns {eval}`core_gap_raw_post:.0f` percentage points per year more than
the rest of the index after the break, against
{eval}`core_gap_raw_pre:.0f` before it. But the middle tier earns
{eval}`adj_gap_raw_post:.0f` points after the break — **more than the AI
core**, from {eval}`adj_gap_raw_pre:.0f` before. Whatever happened at the
break did not order itself by AI intensity, and a model that forces the
middle tier to sit half way between the other two cannot represent this.
Cross-sectional dispersion in the AI core widens by roughly
{eval}`(core_disp_raw - 1) * 100:.0f` per cent. Cumulative abnormal returns
by tier place the separation in time.

In [ ]:
cum = (
    panel.group_by("date", "ai_score")
    .agg(pl.col("xret_pct").mean().alias("day_xret"))
    .sort("date")
    .with_columns(
        pl.col("ai_score").replace_strict(TIER_LABEL).alias("tier"),
        pl.col("day_xret").cum_sum().over("ai_score").alias("cumulative_pp"),
    )
)
rule = pl.DataFrame({"date": [BREAK], "event": ["ChatGPT released"]})

chart = (
    alt.Chart(cum)
    .mark_line()
    .encode(
        x=alt.X("date:T", title=None),
        y=alt.Y("cumulative_pp:Q", title="Cumulative abnormal return (pp)"),
        color=alt.Color("tier:N", sort=list(TIER_LABEL.values()), title=None),
    )
)
vline = alt.Chart(rule).mark_rule(color="#666", strokeDash=[4, 3]).encode(
    x="date:T", tooltip="event:N"
)
(chart + vline).properties(title="Equal-weighted abnormal returns by tier",
                           width=520, height=280)

The lanes part around the break: the adjacent tier and the AI core rise
while the rest of the index drifts down, which is the arithmetic of hedging
against the equal-weighted mean of the same stocks. Three confounds sit inside this picture, and none of them
can be removed by comparing group averages.

**Momentum.** The AI names had already run up before late 2022, and past
winners keep drifting for months. Part of the apparent lane premium is a
momentum premium earned by whatever had risen.

**Volatility persistence.** Technology stocks were more volatile than the
index before 2022 as well. Wider dispersion after the break could simply be
yesterday's volatility continuing, rather than a new regime.

**Calendar turbulence.** March 2020 and the 2022 rate shock widened every
stock's swings for reasons unrelated to AI; a dispersion claim needs to
separate those episodes from a persistent structural shift.

And the dispersion claim needs a different tool regardless: a mean
comparison has no parameter that could even express it.

## One model for both lanes

The response is the **market-hedged excess return** in percentage points,

$$\texttt{xret\_pct}_{it} = 100 \cdot (r_{it} - \hat\beta_{i,t-1}\, m_t),$$

where $r_{it}$ is stock $i$'s daily total return, $m_t$ the equal-weighted
constituent return on day $t$, and $\hat\beta_{i,t-1}$ stock $i$'s trailing
one-year beta as of the previous close. Hedging removes the move common to
all stocks on a day, which sharpens the question to the one actually asked:
not whether the market rose, but whether AI-linked stocks rose *relative to
the typical constituent*. The percentage-point scale matters numerically: the
gradient of the location equation divides by $\sigma^2$, and with
decimal-fraction returns ($\sigma \approx 0.02$) fixed-size boosting steps
overshoot badly. In percent, $\sigma \approx 2$, and the steps behave.

A GaussianLSS additive model gives each stock-day two linked equations
{cite:t}`hofner2014gamboostlss`:

* a **location** equation for the mean excess return $\mu$, identity link;
* a **scale** equation for the daily dispersion $\sigma$, log link —
  coefficients act multiplicatively, so a coefficient of $b$ changes
  volatility by a factor $e^{b}$.

Each equation runs its own component-wise competition: every cycle, the
algorithm adds to each parameter the single base learner that most reduces
that parameter's loss. Which terms end up carrying the location lane and
which the scale lane is decided by the data, not imposed.

### Two design choices the competition depends on

**Each tier gets its own indicator.** `core` and `adj` are separate dummy
columns, with the rest of the index as the reference level, and `core_post`
and `adj_post` are their products with the post-break indicator. These last
two are the difference-in-differences terms: each estimates how much its
tier's gap against the rest of the index moved at the break. Coding the
exposure as a single numeric score instead would force the middle tier's
effect to be exactly half the top tier's — a restriction the raw table
already rejects.

**Every candidate carries two degrees of freedom.** A component-wise
competition compares base learners by how much each reduces the loss, and a
learner with more effective degrees of freedom is selected more often for
reasons unrelated to the signal {cite:t}`hofner2011framework`. An unpenalised
`bols` term has two; a `bbs(x, df=6)` spline has six, and would be selected
ahead of the tier terms on flexibility alone. Each continuous control is
therefore split into a linear term and a centred spline penalised to `df=2`,
so that all ten candidates are compared on equal terms and the non-linear
part is selected only when it reduces the loss by more than the linear part
of the same variable.

The candidate sets:

* **both equations** — `bols(core)`, `bols(adj)`, `bols(post_chatgpt)`,
  `bols(core_post)`, `bols(adj_post)`;
* **location** — momentum, as `bols(mom_lin) + bbs(mom121, center=True, df=2)`;
* **scale** — volatility persistence, as
  `bols(vol_lin) + bbs(vol60_lag, center=True, df=2)`, and calendar time, as
  `bols(block_lin) + bbs(block_idx, center=True, df=2)`, in 28-day blocks
  from January 2019, which absorbs COVID-era and rate-shock turbulence so the
  tier terms are not credited for it.

The location equation has no calendar term because the response cannot carry
one: hedging against the equal-weighted mean of the same stocks pins the
cross-sectional average near zero on every day, so there is no market-wide
time path in $\mu$ left to fit.

**Two specifications, for two different jobs.** `vol60_lag` and `mom121` are
measured on the previous close, which makes them legitimate predictors but not
legitimate controls: after November 2022 an AI stock's trailing volatility is
partly a consequence of the regime under study, so conditioning on it
removes part of the effect along with the confound. The estimation model
therefore uses **baseline** versions — each stock's own volatility and
momentum on the last trading day before the break, held constant over its
whole history. Those are fixed before any treatment, so they can confound the
lane comparison but cannot absorb what the break did. Every lane estimate
below comes from this specification.

The contemporaneous versions are kept for the one job they are right for:
predicting the next day's return, which is how the iteration count is chosen
and how the scale equation's calibration is checked. A baseline control would
be the wrong input there — a stock's volatility in November 2022 is not what a
forecaster would know or want in 2025.

In [ ]:
# Quantile-binned training storage for the spline learners. A P-spline design
# over 930,000 rows is the same 24 columns evaluated at 930,000 points; storing
# one row per bin and a count is exact enough at this resolution and keeps the
# retained design in megabytes rather than gigabytes.
BINS = 100

TIERS = ("bols(core) + bols(adj) + bols(post_chatgpt)"
         " + bols(core_post) + bols(adj_post)")
MOM = f"bols(mom_lin) + bbs(mom121, center=True, df=2, bins={BINS})"
VOL = f"bols(vol_lin) + bbs(vol60_lag, center=True, df=2, bins={BINS})"
CAL = f"bols(block_lin) + bbs(block_idx, center=True, df=2, bins={BINS})"

# Baseline specification: the estimation model. Its controls are firm
# characteristics fixed before the break, so they can confound the lane
# comparison but cannot absorb any of what the break did.
MOM_BASE = f"bols(mom_base_lin) + bbs(mom_base, center=True, df=2, bins={BINS})"
VOL_BASE = f"bols(vol_base_lin) + bbs(vol_base, center=True, df=2, bins={BINS})"
MU_BASE = f"xret_pct ~ {TIERS} + {MOM_BASE}"
SG_BASE = f"xret_pct ~ {TIERS} + {VOL_BASE} + {CAL}"

# Contemporaneous specification: the prediction model. Trailing volatility and
# momentum as of the previous close are the best available predictors, and are
# used as such below; as controls they would absorb the effect being measured.
MU_F = f"xret_pct ~ {TIERS} + {MOM}"
SG_F = f"xret_pct ~ {TIERS} + {VOL} + {CAL}"

split_date = date(2024, 7, 1)
train = pdf[pdf["date"] < pd.Timestamp(split_date)]
valid = pdf[pdf["date"] >= pd.Timestamp(split_date)]

fit_train = mb.gamboostLSS({"mu": MU_F, "sigma": SG_F}, data=train,
                           families=mb.GaussianLSS(),
                           control=mb.boost_control(500, 0.1))
print(f"training path fitted on {len(train):,} rows through "
      f"{train['date'].max().date()}, validation rows {len(valid):,}")

### Choosing the iteration count

Two different questions compete for the stopping rule. The usual one —
which iteration count predicts held-out data best — is answered first, on the
contemporaneous specification and a validation block (July 2024 onward) the
training path never saw. Out-of-sample
loss is the mean Gaussian deviance, $0.5 z^2 + \log \sigma$ with
$z = (y - \mu)/\sigma$, evaluated along the path.

In [ ]:
# The curve is a mean over rows, so it is read on a fixed random sample of the
# validation block rather than all of it: each point costs one prediction per
# parameter, and prediction is charged per row of new data.
probe = valid.sample(n=60_000, random_state=3)
yv = probe["xret_pct"].to_numpy()
grid = list(range(25, 501, 25))
oos = []
for m in grid:
    mu_v = np.asarray(fit_train.predict(parameter="mu", newdata=probe, mstop=m))
    sg_v = np.asarray(fit_train.predict(parameter="sigma", type="response",
                                        newdata=probe, mstop=m))
    oos.append(float(np.mean(0.5 * ((yv - mu_v) / sg_v) ** 2 + np.log(sg_v))))

best_m = grid[int(np.argmin(oos))]
print(f"minimum out-of-sample deviance {min(oos):.5f} at mstop={best_m}")

TIER_TERMS = ("core", "adj", "post_chatgpt", "core_post", "adj_post")
early = {t for picks in fit_train.selected().values() for t in picks[:best_m]}
tier_early = sorted(t for t in early if t in TIER_TERMS)
early_tier_terms = (
    ", ".join(f"`{t}`" for t in tier_early) if tier_early
    else "none of the five tier terms"
)
print(f"selected within the first {best_m} cycles: {early_tier_terms}")

oos_curve = pl.DataFrame({"mstop": grid, "oos_deviance": oos})
line = alt.Chart(oos_curve).mark_line().encode(
    x=alt.X("mstop:Q", title="Boosting iterations"),
    y=alt.Y("oos_deviance:Q", title="Out-of-sample deviance (validation block)",
            scale=alt.Scale(zero=False)),
)
point = alt.Chart(pl.DataFrame({"mstop": [best_m], "oos_deviance": [min(oos)]})
                  ).mark_circle(size=90, color="#d62728").encode(
    x="mstop:Q", y="oos_deviance:Q")
(line + point).properties(
    title="Out-of-time stopping favours a fraction of the path",
    width=520, height=240)

The minimum sits at {eval}`best_m:d` iterations, and by then every tier term
has entered — {eval}`early_tier_terms`. So the usual worry about early
stopping, that the terms carrying the question never get a coefficient, does
not arise here.

The two criteria still disagree, for a subtler reason. Predictive loss asks
which iteration count generalises; the lane question asks whether a
pre-specified contrast has converged. A term can be selected early and still
be far from its converged value, because each selection moves it only one step
of $\nu = 0.1$. Read at {eval}`best_m:d` cycles the two steps are
{eval}`step_core_best:+.1f` and {eval}`step_adj_best:+.1f` percentage points a
year; read at four hundred they are {eval}`head['core_gap_step']:+.1f` and
{eval}`head['adj_gap_step']:+.1f`. The right check is therefore not the
validation curve but the estimate's own path — boost longer and watch whether
the lane estimates settle. Six
scenarios are predicted at each checkpoint, one per tier per era, with the
momentum control at zero, trailing volatility at the panel median, and the
calendar smooth anchored at each era's median block. The location lane has no
calendar term, so its gaps are exact; the scale lane is read *as a ratio*
between tiers within an era, which cancels the anchor.

The quantity plotted is the difference-in-differences itself — the change in
each tier's gap at the break — rather than the two gaps separately. The three
treatment terms are collinear by construction (`core_post` is identically
zero before the break and identically `core` after it), so how the total
post-break gap divides between a standing level and a step is partly a
property of the boosting path. Their difference is the estimand and is what
has to be shown to settle.

In [ ]:
VOL_ANCHOR = float(panel["vol60_lag"].median())
BASE_VOL_ANCHOR = float(panel["vol_base"].median())
PRE_BLOCK = float(panel.filter(pl.col("post_chatgpt") == 0)["block_idx"].median())
POST_BLOCK = float(panel.filter(pl.col("post_chatgpt") == 1)["block_idx"].median())

BASE = {"core": 0.0, "adj": 0.0, "post_chatgpt": 0.0, "core_post": 0.0,
        "adj_post": 0.0, "mom121": 0.0, "vol60_lag": VOL_ANCHOR,
        "block_idx": PRE_BLOCK, "xret_pct": 0.0}


def scenarios():
    """Six matched cells: each tier before and after the break."""
    rows = []
    for post in (0.0, 1.0):
        for tier in ("rest of index", "buildout adjacent", "AI core"):
            r = dict(BASE, post_chatgpt=post)
            if tier == "AI core":
                r["core"], r["core_post"] = 1.0, post
            elif tier == "buildout adjacent":
                r["adj"], r["adj_post"] = 1.0, post
            r["block_idx"] = POST_BLOCK if post else PRE_BLOCK
            r["tier"], r["post"] = tier, post
            rows.append(r)
    out = pd.DataFrame(rows)
    out["mom_lin"] = out["mom121"]
    out["vol_lin"] = out["vol60_lag"]
    out["block_lin"] = out["block_idx"]
    out["mom_base"] = out["mom_base_lin"] = out["mom121"]
    out["vol_base"] = out["vol_base_lin"] = BASE_VOL_ANCHOR
    return out


SCEN = scenarios()


def lane_summary(fit, mstop):
    """Annualised return gap and dispersion ratio, each tier against rest."""
    mu = np.asarray(fit.predict(parameter="mu", newdata=SCEN, mstop=mstop))
    sg = np.asarray(fit.predict(parameter="sigma", type="response",
                                newdata=SCEN, mstop=mstop))
    out = {}
    for k, tier in ((1, "adj"), (2, "core")):
        out[f"{tier}_gap_pre"] = (mu[k] - mu[0]) * 252
        out[f"{tier}_gap_post"] = (mu[k + 3] - mu[3]) * 252
        out[f"{tier}_gap_step"] = out[f"{tier}_gap_post"] - out[f"{tier}_gap_pre"]
        out[f"{tier}_disp_pre"] = sg[k] / sg[0]
        out[f"{tier}_disp_post"] = sg[k + 3] / sg[3]
        out[f"{tier}_disp_step"] = out[f"{tier}_disp_post"] / out[f"{tier}_disp_pre"]
    return out


fit_full = mb.gamboostLSS({"mu": MU_BASE, "sigma": SG_BASE}, data=pdf,
                          families=mb.GaussianLSS(),
                          control=mb.boost_control(400, 0.1))

checks = list(range(20, 401, 20))
paths = [lane_summary(fit_full, m) for m in checks]
step_frame = pl.DataFrame({
    "mstop": checks,
    "AI core": [p["core_gap_step"] for p in paths],
    "buildout adjacent": [p["adj_gap_step"] for p in paths],
})

at_best = lane_summary(fit_full, best_m)
step_core_best, step_adj_best = at_best["core_gap_step"], at_best["adj_gap_step"]
print(f"step at mstop={best_m}: AI core {step_core_best:+.1f}, "
      f"adjacent {step_adj_best:+.1f} pp/yr")

base = alt.Chart(step_frame).transform_fold(
    ["AI core", "buildout adjacent"], as_=["tier", "pp"]
).mark_line().encode(
    x=alt.X("mstop:Q", title="Boosting iterations"),
    y=alt.Y("pp:Q", type="quantitative",
            title="Change in gap against rest of index (pp per year)",
            scale=alt.Scale(zero=True)),
    color=alt.Color("tier:N", title=None),
)
(base + base.mark_point(size=25)).properties(
    title="Difference-in-differences along the boosting path",
    width=520, height=260)

Both step estimates settle well before the path ends, and they settle at
visibly different values. Headline numbers are read at four hundred cycles,
past the plateau.

In [ ]:
head = lane_summary(fit_full, 400)
headline = pl.DataFrame({
    "quantity": [
        "Return gap vs rest of index, before break (pp/yr)",
        "Return gap vs rest of index, after break (pp/yr)",
        "Step at the break (pp/yr)",
        "Dispersion ratio vs rest of index, before break",
        "Dispersion ratio vs rest of index, after break",
    ],
    "AI core": [round(head[f"core_{k}"], 3) for k in
                ("gap_pre", "gap_post", "gap_step", "disp_pre", "disp_post")],
    "buildout adjacent": [round(head[f"adj_{k}"], 3) for k in
                          ("gap_pre", "gap_post", "gap_step", "disp_pre", "disp_post")],
})
display(headline)

sel = fit_full.selected()
alloc = pl.DataFrame([
    {"parameter": param, "term": term, "cycles_selected": count}
    for param, picks in sel.items()
    for term, count in Counter(picks[:400]).most_common()
])
display(alloc.sort("parameter", "cycles_selected", descending=[False, True]))

coefs = fit_full.coef()
coef_rows = [
    {"parameter": param, "term": term,
     "coefficient": round(float(np.ravel(value)[-1]), 4)}
    for param, block in coefs.items()
    for term, value in block.items()
    if term in ("core", "adj", "post_chatgpt", "core_post", "adj_post")
]
display(pl.DataFrame(coef_rows).sort("parameter", "term"))

Both AI tiers stepped up at the break, and the two steps are not the same
size. The buildout-adjacent tier's gap against the rest of the index moves
from {eval}`head['adj_gap_pre']:.1f` percentage points a year to
{eval}`head['adj_gap_post']:.1f`, a step of
{eval}`head['adj_gap_step']:+.1f`. The AI core moves from
{eval}`head['core_gap_pre']:.1f` to {eval}`head['core_gap_post']:.1f`, a step
of {eval}`head['core_gap_step']:+.1f` on top of an advantage it already had.
The tier that sells power, connectivity and capacity into the buildout gained
more at the break than the tier that builds the compute.

Dispersion separates the tiers the other way. The AI core runs
{eval}`(head['core_disp_pre'] - 1) * 100:.0f` per cent wider than the rest of
the index before the break and
{eval}`(head['core_disp_post'] - 1) * 100:.0f` per cent after, while the
adjacent tier moves from {eval}`(head['adj_disp_pre'] - 1) * 100:.0f` to
{eval}`(head['adj_disp_post'] - 1) * 100:.0f` per cent. The scale step belongs
to the AI core alone: its stocks did not merely earn more, they became harder
to predict individually, and the tier with the larger return step did not.

The coefficient table gives each tier term on its own scale — percentage
points per day in the location equation, log-multiplicative in the scale
equation, so a coefficient of $b$ multiplies daily volatility by $e^{b}$. The
allocation table shows how few cycles the scale lane spends on the tier terms
once calendar turbulence is priced, which is what the next two sections
examine.

### One control sits on the causal path

Refitting with each control added in turn shows how much of the raw pattern
each one carries. The last row is the specification the case study argues
against, and it is included because it is the one most analyses would reach
for.

Adding contemporaneous volatility and momentum in place of their baseline
values changes the estimate for a reason that has nothing to do with
confounding. Trailing volatility after November 2022 sits on the causal path
between being an AI stock and the outcome: the break moved it, and it then
moves the response. Conditioning on a mediator estimates the **direct
effect**, the part of the lane premium that does not travel through
volatility and momentum. That is a different quantity from the total effect,
and neither is a correction of the other.

In [ ]:
VARIANTS = [
    ("tier terms only",
     f"xret_pct ~ {TIERS}", f"xret_pct ~ {TIERS}"),
    ("+ pre-break volatility and momentum (headline)", MU_BASE, SG_BASE),
    ("+ contemporaneous volatility and momentum", MU_F, SG_F),
]

ablation = []
for label, mu_f, sg_f in VARIANTS:
    if label.endswith("(headline)"):
        v = head
    else:
        vfit = mb.gamboostLSS({"mu": mu_f, "sigma": sg_f}, data=pdf,
                              families=mb.GaussianLSS(),
                              control=mb.boost_control(400, 0.1))
        v = lane_summary(vfit, 400)
        del vfit
    ablation.append({
        "specification": label,
        "AI core step (pp)": round(v["core_gap_step"], 1),
        "adjacent step (pp)": round(v["adj_gap_step"], 1),
        "AI core dispersion after": round(v["core_disp_post"], 2),
        "adjacent dispersion after": round(v["adj_disp_post"], 2),
    })

display(pl.DataFrame(ablation))

total_core, total_adj = ablation[0]["AI core step (pp)"], ablation[0]["adjacent step (pp)"]
base_core, base_adj = ablation[1]["AI core step (pp)"], ablation[1]["adjacent step (pp)"]
direct_core, direct_adj = ablation[2]["AI core step (pp)"], ablation[2]["adjacent step (pp)"]
total_disp = ablation[0]["AI core dispersion after"]
base_disp = ablation[1]["AI core dispersion after"]
direct_disp = ablation[2]["AI core dispersion after"]

With tier terms alone, the break moves the AI-core gap by
{eval}`total_core:.1f` percentage points a year and the adjacent gap by
{eval}`total_adj:.1f`. Adding the calendar smooth and the baseline
volatility and momentum controls — the headline specification — leaves
{eval}`base_core:.1f` and {eval}`base_adj:.1f`. Those two controls are
genuine confounders: high-volatility firms were high-volatility firms before
anyone had heard of ChatGPT, and the AI tiers were over-represented among
them.

Replacing them with their contemporaneous versions leaves
{eval}`direct_core:.1f` and {eval}`direct_adj:.1f`, removing about four fifths
of the AI core's step and about a sixth of the adjacent tier's. Read as a
confound adjustment, that row would say the AI core's return step was mostly
an artefact; read correctly, it says that most of the AI core's return step
travelled through its own volatility, and the buildout tier's did not. The direct effects are real quantities,
but they answer "what did the break do that did not run through volatility",
which is not the question the two-lane hypothesis asks.

The scale lane shows the same ordering. The AI-core dispersion ratio after
the break falls from {eval}`total_disp:.2f` with tier terms alone to
{eval}`base_disp:.2f` under the headline specification and
{eval}`direct_disp:.2f` once the contemporaneous control enters. Calendar
turbulence and pre-existing firm volatility together account for most of the
raw widening; what survives is a genuine, and much smaller, AI-core-specific
step.

### Selection frequencies across resamples

Every number above is a point estimate from one path on one panel. The
dispersion step in particular rests on the tier terms being selected in a
small number of cycles out of the sixteen candidates the two equations put up
between them, which is exactly the situation stability selection was built for
{cite:t}`meinshausen2010stability,shah2013variable`: refit on many random
half-samples, record how often each candidate is selected, and keep only
those whose selection frequency clears a cutoff derived from a bound on the
per-family error rate.

For an LSS model the resampling is run on the **non-cyclic** fitting
algorithm {cite:t}`thomas2018gradient`, which updates one parameter per
iteration rather than all of them in turn. That makes "the first $q$ terms
selected" a well-defined quantity across the $\mu$ and $\sigma$ equations
jointly, which is what the error bound needs.

Twenty-five complementary pairs — fifty half-sample refits, since the error
bound is derived on each half and its complement — of a
nine-hundred-thousand-row panel is more computation than this page is worth,
so the resampling runs on every third trading day. Thinning the calendar costs statistical power and nothing else —
the cross-section is untouched, and daily excess returns carry little
day-to-day autocorrelation — so a term that clears the cutoff here would also
clear it on the full panel.

In [ ]:
STABSEL_B = 25
# `q` is the number of distinct learners each half-sample is allowed to
# select, and it has to be a number the half-samples actually reach: a fold
# that runs out of iterations first contributes a shorter sequence than the
# error bound assumes. On this specification they do not go far. Sixteen
# candidates compete, nine in the scale equation and seven in the location
# equation, and a non-cyclic path spends most of its steps re-selecting the
# few scale terms already in — at three hundred iterations every half-sample
# reaches five distinct learners, thirty-nine of fifty reach six, and none
# reaches eight.
#
# `mstop` is an upper bound rather than a budget every fold spends: each fold
# stops at its fifth distinct learner, which is why three hundred iterations
# cost less here than eighty did at an unreachable `q=8`.
STABSEL_MSTOP = 300
STABSEL_Q = 5
day_index = (
    panel.select("date").unique().sort("date").with_row_index("day_number")
)
thinned = (
    panel.join(day_index, on="date")
    .filter(pl.col("day_number") % 3 == 0)
    .drop("day_number")
    .to_pandas()
)
print(f"stability selection on {len(thinned):,} rows "
      f"({thinned['date'].nunique()} trading days)")

fit_nc = mb.gamboostLSS({"mu": MU_BASE, "sigma": SG_BASE}, data=thinned,
                        families=mb.GaussianLSS(), method="noncyclic",
                        control=mb.boost_control(STABSEL_MSTOP, 0.1))
stab = fit_nc.stabsel(q=STABSEL_Q, PFER=2.0, B=STABSEL_B)

freq = np.asarray(stab.max_)
stab_frame = pl.DataFrame({
    "learner": list(stab.learner_names),
    "frequency": freq,
}).with_columns(
    pl.col("learner").str.split(".").list.last().alias("parameter"),
).filter(~pl.col("learner").str.contains(r"\(offset\)|\(Intercept\)")
         ).sort("frequency", descending=True)
stab_cutoff = float(stab.cutoff)
print(f"a candidate survives at a selection frequency of {stab_cutoff:.2f} "
      f"or above (q={STABSEL_Q}, PFER=2, B={STABSEL_B}, "
      f"{stab.n_fits} half-sample fits)")
display(stab_frame.head(12))

bars = alt.Chart(stab_frame).mark_bar(color="#4e79a7").encode(
    x=alt.X("frequency:Q", title="Selection frequency",
            scale=alt.Scale(domain=[0, 1])),
    y=alt.Y("learner:N", title=None, sort="-x"),
    tooltip=[alt.Tooltip("learner:N"),
             alt.Tooltip("frequency:Q", format=".2f")],
)
cut = alt.Chart(pl.DataFrame({"cutoff": [stab_cutoff]})).mark_rule(
    color="#e15759", strokeDash=[4, 3]).encode(x="cutoff:Q")
display((bars + cut).properties(
    title="Selection frequency over half-samples", width=440, height=320))

del fit_nc, thinned

TIER_TERMS = ("core", "adj", "post_chatgpt", "core_post", "adj_post")
survivors = stab_frame.filter(pl.col("frequency") >= stab_cutoff)
survivor_names = ", ".join(f"`{n}`" for n in survivors["learner"])
tier_survivors = ", ".join(
    f"`{n}`" for n in survivors["learner"]
    if n.split(".")[0] in TIER_TERMS) or "none"


def stab_freq(learner):
    row = stab_frame.filter(pl.col("learner") == learner)
    return float(row["frequency"][0]) if row.height else 0.0


core_post_sigma = stab_freq("core_post.sigma")
adj_post_sigma = stab_freq("adj_post.sigma")
print(f"survives: {survivor_names}")
print(f"tier terms among them: {tier_survivors}")

The scale equation's two treatment terms, `core_post.sigma` and
`adj_post.sigma`, are selected in {eval}`core_post_sigma:.0%` and
{eval}`adj_post_sigma:.0%` of half-samples and clear the cutoff of
{eval}`stab_cutoff:.2f`. They are two of the five terms the scale equation
selects, and every half-sample selects the same five: the two treatment terms,
baseline volatility, and the two calendar terms. The dispersion result is not
an artefact of a handful of cycles on one panel — under resampling it is as
firm as calendar time, which is the one thing on this page nobody would
dispute belongs in the scale equation.

The location terms do not clear the cutoff, and the reason is structural
rather than a verdict on their size. Non-cyclic boosting spends each iteration
on whichever parameter offers the larger loss reduction, and `q` counts
distinct learners across both equations together. The scale equation has more
to gain here, so it takes all five selections in every half-sample and no
location term enters the recorded window at all — every one of them reads a
frequency of zero, which is what a term that was never given the chance looks
like, not what an unstable term looks like. Running stability selection on the
location equation alone, with its own `q`, would be the way to ask that
question separately.

### Tail weight as a third equation

Everything so far assumes a Gaussian response. Daily equity returns are not
Gaussian, and not marginally so: the standardised response here has an excess
kurtosis far outside anything a normal law admits. This matters more for an
LSS fit than for a mean regression, because the scale equation's gradient is
driven by $z^2$ — a handful of extreme days steer $\sigma$, and the family
insists those days cannot happen.

`StudentTLSS` adds a third equation for the degrees-of-freedom parameter
$\nu$, which governs tail weight: small $\nu$ means frequent extreme moves,
and $\nu \to \infty$ recovers the Gaussian. Fitting it does two things. It
checks whether the dispersion result survives a family that can absorb the
tails in a separate parameter, and it lets the tails themselves be a lane
outcome rather than a nuisance.

In [ ]:
excess_kurtosis = float(
    ((panel["xret_pct"] - panel["xret_pct"].mean()) ** 4).mean()
    / panel["xret_pct"].std() ** 4 - 3
)
print(f"excess kurtosis of the response: {excess_kurtosis:.0f}")

fit_t = mb.gamboostLSS(
    {"mu": MU_BASE, "sigma": SG_BASE, "df": f"xret_pct ~ {TIERS}"},
    data=pdf, families=mb.StudentTLSS(),
    control=mb.boost_control(300, 0.1))

nu = np.asarray(fit_t.predict(parameter="df", type="response",
                              newdata=SCEN, mstop=300))
mu_t = np.asarray(fit_t.predict(parameter="mu", newdata=SCEN, mstop=300))
sg_t = np.asarray(fit_t.predict(parameter="sigma", type="response",
                                newdata=SCEN, mstop=300))

student = pl.DataFrame({
    "tier": SCEN["tier"].tolist(),
    "era": ["before break"] * 3 + ["after break"] * 3,
    "return (pp/yr)": np.round(mu_t * 252, 1),
    "sigma (pp/day)": np.round(sg_t, 3),
    "nu (tail weight)": np.round(nu, 2),
})
display(student)

sel_t = fit_t.selected()
nu_alloc = Counter(sel_t["df"][:300]).most_common(6)
print("degrees-of-freedom equation, cycles per term:",
      ", ".join(f"{t} {c}" for t, c in nu_alloc))

t_step_core = float((mu_t[5] - mu_t[3]) - (mu_t[2] - mu_t[0])) * 252
t_step_adj = float((mu_t[4] - mu_t[3]) - (mu_t[1] - mu_t[0])) * 252
t_disp_core = float((sg_t[5] / sg_t[3]) / (sg_t[2] / sg_t[0]))
nu_core_pre, nu_core_post = float(nu[2]), float(nu[5])
nu_rest_pre, nu_rest_post = float(nu[0]), float(nu[3])

The response has an excess kurtosis of {eval}`excess_kurtosis:.0f`; a Gaussian
law has zero. Under `StudentTLSS` the fitted tail index runs between
{eval}`min(nu_core_post, nu_rest_pre):.1f` and
{eval}`max(nu_core_post, nu_rest_pre):.1f` degrees of freedom, low enough that
the fourth moment barely exists — daily excess returns of this kind are not
approximately normal at any sample size.

The dispersion result survives the change of family: the AI core's dispersion
ratio still steps up at the break, by a factor of
{eval}`t_disp_core:.3f` relative to the other tiers' move, and the return
steps keep their ordering, {eval}`t_step_core:+.1f` percentage points a year
for the AI core against {eval}`t_step_adj:+.1f` for the adjacent tier.

The $\nu$ equation adds a result the Gaussian model has no parameter for.
Tail weight is not a fixed property of the panel: it moves with the tiers and
with the break. The AI core's fitted $\nu$ falls from
{eval}`nu_core_pre:.2f` before the break to {eval}`nu_core_post:.2f` after,
while the rest of the index moves from {eval}`nu_rest_pre:.2f` to
{eval}`nu_rest_post:.2f`. A lower $\nu$ at the same $\sigma$ means the same
typical day with fatter extremes. The AI lane did not only get wider after the
break; the shape of its bad days changed, and that change is invisible to any
two-parameter fit.

### Calibration of the fitted dispersions

An LSS fit is only as good as its scale equation, and checking it needs a
residual that is family-aware. The **randomised quantile residual**
{cite:t}`dunn1996randomized` is the standard construction: evaluate each
observation's own fitted distribution function at the observed value, then
map the result through the inverse normal. If the fitted distribution is
right — location, scale, *and* shape — these residuals are standard normal,
whatever family produced them.

That property is what makes the check meaningful. A raw standardised residual
$z = (y - \mu)/\sigma$ has unit variance only if the family's variance
assumption holds, so any summary of $|z|$ under a Gaussian fit to
fat-tailed data measures the misspecification rather than the calibration.
Binning quantile residuals by predicted $\sigma$ and reading their standard
deviation per bin gives a chart whose reference line is 1.0 by construction.

The validation block provides the out-of-time check, at two iteration
counts: the prediction-optimal one and the four hundred cycles the headline
uses.

In [ ]:
def calibration_bins(fit, mstop, label):
    sg = np.asarray(fit.predict(parameter="sigma", type="response",
                                newdata=probe, mstop=mstop))
    qr = np.asarray(mb.quantile_residuals(fit, newdata=probe, mstop=mstop,
                                          random_state=11))
    frame = pl.DataFrame({"sigma_pred": sg, "qr": qr})
    frame = frame.with_columns(
        (pl.col("sigma_pred").rank("ordinal") * 10 // (len(frame) + 1))
        .clip(0, 9).alias("decile")
    )
    return (
        frame.group_by("decile")
        .agg(pl.col("sigma_pred").mean().alias("predicted_sigma"),
             pl.col("qr").std().alias("residual_sd"))
        .sort("decile")
        .with_columns(pl.lit(label).alias("fit"))
    )


fit_train_t = mb.gamboostLSS(
    {"mu": MU_F, "sigma": SG_F, "df": f"xret_pct ~ {TIERS}"},
    data=train, families=mb.StudentTLSS(),
    control=mb.boost_control(300, 0.1))

cal = pl.concat([
    calibration_bins(fit_train, best_m, f"Gaussian, mstop={best_m}"),
    calibration_bins(fit_train, 400, "Gaussian, mstop=400"),
    calibration_bins(fit_train_t, 300, "Student-t, mstop=300"),
])

pts = alt.Chart(cal).mark_circle(size=55).encode(
    x=alt.X("predicted_sigma:Q", title="Predicted sigma (pp per day)"),
    y=alt.Y("residual_sd:Q", title="SD of quantile residuals",
            scale=alt.Scale(zero=False)),
    color=alt.Color("fit:N", title=None),
    tooltip=[alt.Tooltip("predicted_sigma:Q", format=".2f"),
             alt.Tooltip("residual_sd:Q", format=".3f"),
             alt.Tooltip("fit:N")],
)
perfect = alt.Chart(pl.DataFrame({"y": [1.0]})).mark_rule(
    color="#d62728", strokeDash=[4, 3]).encode(y="y:Q")
(pts + perfect).properties(
    title="Scale calibration on the validation block", width=480, height=300)

cal_summary = (
    cal.group_by("fit")
    .agg(pl.col("residual_sd").mean().round(3).alias("mean SD"),
         (pl.col("residual_sd").max() - pl.col("residual_sd").min())
         .round(3).alias("spread across deciles"))
    .sort("fit")
)
display(cal_summary)


def cal_stat(label, column):
    return float(cal_summary.filter(pl.col("fit") == label)[column][0])


gauss_mean = cal_stat("Gaussian, mstop=400", "mean SD")
gauss_spread = cal_stat("Gaussian, mstop=400", "spread across deciles")
gauss_early_mean = cal_stat(f"Gaussian, mstop={best_m}", "mean SD")
t_mean = cal_stat("Student-t, mstop=300", "mean SD")
t_spread = cal_stat("Student-t, mstop=300", "spread across deciles")

# What the Gaussian-consistent scale estimator would have reported on
# residuals this fat-tailed, had it been used instead.
z_valid = ((probe["xret_pct"].to_numpy()
            - np.asarray(fit_train.predict(parameter="mu", newdata=probe, mstop=400)))
           / np.asarray(fit_train.predict(parameter="sigma", type="response",
                                          newdata=probe, mstop=400)))
naive_ratio = float(np.mean(np.abs(z_valid)) * np.sqrt(np.pi / 2) / np.std(z_valid))
print(f"mean|z|*sqrt(pi/2) divided by sd(z) on the validation block: "
      f"{naive_ratio:.3f}")

The scale estimates are close to the residual-based reference out of sample. Quantile residuals from
the Gaussian fit have a standard deviation averaging
{eval}`gauss_mean:.3f` across the predicted-$\sigma$ deciles, against the
reference of 1.0, and the Student-t fit averages {eval}`t_mean:.3f` with the
smaller spread across deciles ({eval}`t_spread:.3f` against
{eval}`gauss_spread:.3f`). Fitted dispersions show no consistent upward or downward deviation on 2024–26 data, and the iteration count barely moves
the answer: at the prediction-optimal {eval}`best_m:d` cycles the mean is
{eval}`gauss_early_mean:.3f`, so the four hundred cycles the headline uses are
not overfitting the scale equation.

The choice of residual is what makes that reading possible. Summarising raw
standardised residuals with the Gaussian-consistent estimator
$\bar{|z|}\sqrt{\pi/2}$ would report {eval}`naive_ratio:.3f` times the
residuals' own standard deviation on this validation block, a shortfall of
{eval}`(1 - naive_ratio) * 100:.0f` per cent produced entirely by the
kurtosis, before any question of model fit arises. Against a reference line drawn at 1.0, that estimator would have
made a well-calibrated scale model look uniformly over-confident, and the
mistake would have been invisible: the curve has the right shape and only the
level is wrong.

## Limitations

**The break date is ex post.** November 2022 was chosen because the narrative
chose it. Cumulative abnormal returns by tier separate over months rather
than on a day, and a different plausible break — the January 2025 efficiency
scare, say — would give different step sizes.

**Beta hedging is estimated**, with error correlated across days for the same
stock. Stability selection checks which terms survive resampling, but it is
not a confidence interval: standard errors on the step sizes would need
a block bootstrap that respects both the time-series and cross-sectional
dependence, which is not reported here.

**The response is zero-sum.** Hedging against the equal-weighted mean of the
same constituents means the tiers' gaps are relative by construction. The two
AI tiers are {eval}`ai_share * 100:.0f` per cent of the equal-weighted index,
so they are in part hedged against themselves; a tier's measured advantage is
against the other tiers, never against cash.

**Exposure is industry-level**, so the estimate is an intent to treat on
being an AI-adjacent business, diluted by firms in classified industries with
little AI revenue.

**Survivorship and composition** both bound the panel. The constituent list
is today's index projected backwards, and the late entrants excluded to keep
the comparison within-firm are exactly the names most exposed to the
buildout.

## Summary

Fitting location and scale jointly turns the two-lane narrative into three
measured quantities, and they do not point the same way. Both AI tiers gained
against the rest of the index at the break — {eval}`base_adj:.1f` percentage
points a year for the buildout-adjacent tier and {eval}`base_core:.1f` for the
AI core, once firm characteristics fixed before the break are controlled for.
Only the AI core's dispersion stepped up, to
{eval}`(head['core_disp_post'] - 1) * 100:.0f` per cent above the rest of the
index. And under a Student-t fit the AI core's tail index falls furthest, from
{eval}`nu_core_pre:.2f` to {eval}`nu_core_post:.2f` against a rest-of-index
lane that barely moves: the tier that gained least in return is the one whose
distribution changed shape most.

Two choices decide those numbers, and both are easy to get wrong. Coding
exposure as a single score forces the middle tier's effect to be half the top
tier's, which reverses the ordering the data shows. And controlling for
trailing volatility measured after the break removes most of the AI core's
return step, because that volatility is a consequence of the break rather than
a confound of it — a control chosen for its predictive value, used for a
causal question.

The machinery generalises wherever two claims of this form travel together.
[](motorcycle_variance.ipynb) shows the single-equation version of the scale
problem, [](choosing_a_distribution.ipynb) treats the choice of family as a
question the data can answer, [](growth_centiles.ipynb) fits richer families
where the centile is the deliverable, and [](stopping_rule.ipynb) treats the
tension between a prediction-optimal iteration count and an estimation target
at length.

```{bibliography}
:filter: docname in docnames
```